# Solving a Quadratic Unconstrained Binary Optimization instance

Solving a QUBO instance is straightforward with `qubo-solver`. We can directly use the `Solver` class by providing a `Instance` with a given `SolverConfig` configuration.
`SolverConfig` specifies whether to use a classical approach or a quantum one. Note that `SolverConfig` comes with many options but the default ones can be used straightforwardly.
We have however more advanced tutorials on the quantum-related components to dive deeper into these advanced concepts.

## Solving with a quantum approach

To use a quantum approach, several choices have to be made regarging the configuration, explained in more details in the [`SolverConfig` section of the documentation](https://pasqal-io.github.io/qubo-solver/latest/content/solver/).

One main decision is about the [backend](https://pasqal-io.github.io/qubo-solver/latest/content/backend/), that is how we choose to perform quantum runs. We can decide to either perform our on emulators (locally, or remotely) or using a real quantum processing unit (QPU). Our QPU, based on the Rydberg Analog Model, is accessible remotely.

### Available backend types and devices

The supported backends are available via [`Qooqit`](https://pasqal-io.github.io/qoolqit/latest/api/qoolqit/execution/backends/), a Python package designed for algorithm development in the Rydberg Analog Model.

The backends can be divided into 3 main categories:

- [Local emulators](https://pasqal-io.github.io/emulators/latest/) (Qutip, Emu_mps, Emu_sv, ...),
- [Remote emulators]((https://docs.pasqal.com/cloud/emu-tn/)), which can be accessed via [`pasqal_cloud`](https://docs.pasqal.com/cloud/),
- [A remote QPU, such as Fresnel](https://docs.pasqal.com/cloud/fresnel-job/).

A backend will use device specifications to perform quantum computations. The list of supported devices can be found in [`the QoolQit devices documentation`](https://pasqal-io.github.io/qoolqit/latest/api/qoolqit/devices/).

### Running locally with an emulator

We can perform quantum simulations locally via an emulator (here, we choose the `QUTIP` emulator by default).

In [3]:
from qubosolver import Instance, solvers, Solver, matrix, LocalEmulator, analysis

# define QUBO
Q = matrix.tensor([[-0.2, 0, 1.0], [0, 0,1.5], [1.0, 1.5, 0]])
instance = Instance(matrix=Q)

# Create a SolverConfig object to use a quantum backend
quantum_config = solvers.QuantumConfig(backend=LocalEmulator())
config = solvers.Config(solving=quantum_config)

# Instantiate the quantum solver.
solver = Solver(instance, config)

# Solve the QUBO problem.
solution = solver.solve()

# Display results
print(analysis.to_dataframe([solution]))

  labels bitstrings  costs  counts  probs
0      0        100   -0.2   154.0  0.154
1      0        110   -0.2    51.0  0.051
2      0        000    0.0   120.0  0.120
3      0        001    0.0   360.0  0.360
4      0        010    0.0   315.0  0.315


### Running with a remote connection

We can decide to perform our runs remotely via [`pasqal_cloud`](https://docs.pasqal.com/cloud/).
To do so, we have to provide several information after [setting up an account](https://docs.pasqal.com/cloud/set-up/).

#### On a real QPU

The code above can be modified to solve the QUBO instance using our real QPU remotely as follows (run only with your `pasqal_cloud` information):

In [ ]:
from qubosolver import Instance, solvers, Solver, matrix, analysis
import qoolqit
from qoolqit.execution import QPU
from pasqal_cloud import PasqalCloudConnection

# Replace with your username, project id and password on the Pasqal Cloud.
USERNAME="#TO_PROVIDE"
PROJECT_ID="#TO_PROVIDE"
PASSWORD=None

if PASSWORD is not None:

    # define QUBO
    Q = matrix.tensor([[-0.2, 0, 1.0], [0, 0,1.5], [1.0, 1.5, 0]])
    instance = Instance(matrix=Q)

    connection = PasqalCloudConnection(
        username=USERNAME,
        password=PASSWORD,
        project_id=PROJECT_ID,
    )
    # Get available devices
    print(f"Available devices: {connection.fetch_available_devices()}")
    # Choose a device
    device = qoolqit.Device.from_connection(connection, "FRESNEL_CAN1")
    quantum_config = solvers.QuantumConfig(device=device, backend=QPU(connection=connection, num_shots=1000))
    config = solvers.Config(solving=quantum_config)
    # Run the solver
    solver = Solver(instance, config)
    solution = solver.solve()

    # Display results
    print(analysis.to_dataframe([solution]))

#### On a remote emulators

Emulators are also available remotely via `pasqal_cloud`:

In [ ]:
from qubosolver import Instance, solvers, Solver, matrix, RemoteEmulator, analysis
from pasqal_cloud import PasqalCloudConnection

# Replace with your username, project id and password on the Pasqal Cloud.
USERNAME="#TO_PROVIDE"
PROJECT_ID="#TO_PROVIDE"
PASSWORD=None

if PASSWORD is not None:

    # define QUBO
    Q = matrix.tensor([[-0.2, 0, 1.0], [0, 0,1.5], [1.0, 1.5, 0]])
    instance = Instance(matrix=Q)

    connection = PasqalCloudConnection(
        username=USERNAME,
        password=PASSWORD,
        project_id=PROJECT_ID,
    )
    quantum_config = solvers.QuantumConfig(backend=RemoteEmulator(connection=connection))
    config = solvers.Config(solving=quantum_config)
    # Run the solver
    solver = Solver(instance, config)
    solution = solver.solve()

    # Display results
    print(analysis.to_dataframe([solution]))

## Solving with a classical approach

We show below an example of solving a QUBO using CPLEX.
More information on classical approaches can be found in the `Classical solvers` section of the `Contents` documentation.

In [7]:
from qubosolver import Instance, Solver, solvers, matrix, analysis

# define QUBO
Q = matrix.tensor([[-0.2, 0, 1.0], [0, 0,1.5], [1.0, 1.5, 0]])
instance = Instance(matrix=Q)

# Create a SolverConfig object with classical solver options.
classical_config = solvers.ClassicalConfig(
    algorithm="tabu_search",
    tabu_time_limit=10.0,
)
config = solvers.Config(solving=classical_config)

# Instantiate the classical solver via the pipeline's classical solver dispatcher.
classical_solver = Solver(instance, config)

# Solve the QUBO problem.
solution = classical_solver.solve()

# Display results
print(analysis.to_dataframe([solution]))

  labels bitstrings  costs  counts  probs
0      0        110   -0.2     1.0    1.0


# Using the functional API

One can also use the functional API

## Quantum approach

In [ ]:
from qubosolver import Instance, solvers, matrix, LocalEmulator, analysis, embedding, drive_shaping, Solution
import qoolqit

# define QUBO
Q = matrix.tensor([[-0.2, 0, 1.0], [0, 0,1.5], [1.0, 1.5, 0]])
instance = Instance(matrix=Q)

# You can use the same devices and backends as in the Object API (Remote, QPU)
device = qoolqit.AnalogDevice()
backend = LocalEmulator()
register = embedding.blade.embed(instance)
drive = drive_shaping.proportional_diagonal.build_drive(instance, register, device=device)
job = solvers.analog_quantum_sampling(register, drive, backend=backend, device=device)
solution = Solution.from_results(job.results(), instance=instance)

# Display results
print(analysis.to_dataframe([solution]))

  labels bitstrings  costs  counts  probs
0      0        110   -0.2   465.0  0.465
1      0        100   -0.2    51.0  0.051
2      0        010    0.0    49.0  0.049
3      0        000    0.0   156.0  0.156
4      0        001    0.0   279.0  0.279


This is slightly more verbose than the Object API, but gives better insight and control. Some things are only possible with the functional API


## Classical Approach

In [12]:
from qubosolver import Instance, solvers, matrix, analysis, bitstrings, torch_rng

# define QUBO
Q = matrix.tensor([[-0.2, 0, 1.0], [0, 0,1.5], [1.0, 1.5, 0]])
instance = Instance(matrix=Q)
solution = solvers.tabu_search(instance, bitstrings.rand(1, instance.size, rng=torch_rng(15)), time_limit=10.0)

# Display results
print(analysis.to_dataframe([solution]))

  labels bitstrings  costs  counts  probs
0      0        110   -0.2     1.0    1.0
